# Working with multidimensional climate data using Python (Part 1)

In this notebook, I will go through the basics of xarray and other necessary packages that are used to explore and analyze multidimensional climate datasets (**CMIP6, ERA5, CRU, CHRIPS etc.**)

This notebook is structured as follows:
1. Reading dataset
1. Inspecting the dataset
1. filtering by time and location

In [4]:
import xarray as xr
import numpy as np
import pandas as pd

**The main xarray objects**<br>
>DataArray: one labeled variable, for example daily precipitation or tasmax.

>Dataset: a container holding one or more variables, coordinates, and metadata.

>Dimensions: axes of an array, often time, latitude, longitude, or level.

>Coordinates: labels that identify positions along dimensions, such as dates and grid-cell locations.

>Attributes (attrs): metadata such as units, long name, and source.

## 0. Introduction to necessary file formats: 
**About NetCDF File:**<br>
>***NetCDF (Network Common Data Form)*** is a file format designed to store scientific data, especially multidimensional data such as temperature, precipitation, humidity, and wind speed over time and geographical locations.<br>
- Structure: Stores data in dimensions (e.g., time, latitude, longitude), variables, and metadata.
- Common uses: Climate model outputs, satellite observations, and reanalysis datasets such as ERA5.
- Python tools: xarray, netCDF4, and h5netcdf.<br>

**About GRIB File:**<br>
>***GRIB (GRIdded Binary)*** is a compact binary file format developed by the World Meteorological Organization (WMO) for storing and transmitting meteorological and weather forecast data.<br>
- Structure: Stores gridded weather variables along with information about time, geographical coordinates, forecast steps, and data levels.
- Common uses: Numerical weather prediction, weather forecasts, and meteorological reanalysis datasets.
- Python tools: cfgrib with xarray, and eccodes.

## 1. Reading Datasets

### 1.1 Opening a single netcdf file

In [2]:
temp_df = xr.open_dataset("D:\CMIP6 Project\Thesis work\Data Analysis\Data\ERA5_Data\ERA5_Daily_Tmin_Tmax_1989_2019.nc")
temp_df

c:\Users\Anwoy\miniconda3\envs\cmip6\Lib\site-packages\xarray\backends\plugins.py:110: RuntimeWarning: Engine 'cfgrib' loading failed:
Cannot find the ecCodes library
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)


<xarray.Dataset> Size: 75MB
Dimensions:     (valid_time: 11322, latitude: 33, longitude: 25)
Coordinates:
  * latitude    (latitude) float64 264B 28.0 27.75 27.5 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 200B 87.0 87.25 87.5 ... 92.5 92.75 93.0
  * valid_time  (valid_time) datetime64[ns] 91kB 1989-01-01 ... 2019-12-31
Data variables:
    tasmax      (valid_time, latitude, longitude) float32 37MB ...
    tasmin      (valid_time, latitude, longitude) float32 37MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-02-18T15:22 GRIB to CDM+CF via cfgrib-0.9.1...

### 1.2 Opening multiple datasets

In [6]:
# Files should have compatible dimensions and coordinates
ds = xr.open_mfdataset(
    r"D:\CMIP6 Project\Thesis work\Data Analysis\Data\ERA5_Data\tasmax\era5_max*.nc",
    combine="by_coords"
)

### 1.3 Opening GRIB file

```python
import cfgrib
data = cfgrib.open_datasets(file_location)
```
**Note: If you face an error while importing cfgrib, use this code.**<br>
```python
import os
import sys
from pathlib import Path

env = Path(sys.prefix)
dll_dir = env / "Library" / "bin"

print("DLL directory exists:", dll_dir.exists())
print("ecCodes DLL exists:", (dll_dir / "eccodes.dll").exists())

if dll_dir.exists():
    os.environ["PATH"] = str(dll_dir) + os.pathsep + os.environ.get("PATH", "")
    if hasattr(os, "add_dll_directory"):
        os.add_dll_directory(str(dll_dir))

import eccodes
print("ecCodes version:", eccodes.codes_get_api_version())
```

## 2. Inspecting the dataset

In [10]:
print(temp_df.data_vars)        # available data variables
print(temp_df.coords)           # coordinate names and values
print(temp_df.dims)             # dimension sizes
print(temp_df.attrs)            # global metadata
print(temp_df["tasmax"].attrs)      # metadata for one variable, if it is named tp

Data variables:
    tasmax   (valid_time, latitude, longitude) float32 37MB ...
    tasmin   (valid_time, latitude, longitude) float32 37MB ...
Coordinates:
  * latitude    (latitude) float64 264B 28.0 27.75 27.5 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 200B 87.0 87.25 87.5 ... 92.5 92.75 93.0
  * valid_time  (valid_time) datetime64[ns] 91kB 1989-01-01 ... 2019-12-31
FrozenMappingWarningOnValuesAccess({'valid_time': 11322, 'latitude': 33, 'longitude': 25})
{'GRIB_centre': 'ecmf', 'GRIB_centreDescription': 'European Centre for Medium-Range Weather Forecasts', 'GRIB_subCentre': 0, 'Conventions': 'CF-1.7', 'institution': 'European Centre for Medium-Range Weather Forecasts', 'history': '2026-02-18T15:22 GRIB to CDM+CF via cfgrib-0.9.15.1/ecCodes-2.42.0 with {"source": "2m_temperature.grib", "filter_by_keys": {"stream": ["oper"]}, "encode_cf": ["parameter", "time", "geography", "vertical"]}\nearthkit.transforms.aggregate.temporal.daily_reduce(2m_temperature_stream-oper, how=m

In [13]:
# List variable names
list(temp_df.data_vars)

# Inspect a variable's dimensions, units, and missing values
var = temp_df["tasmax"]              # replace tasmax with your variable name
print(var.dims)
print(var.coords)
print(var.shape)
print(var.dtype)
print(var.isnull().sum().compute())  # compute if using Dask/lazy data

('valid_time', 'latitude', 'longitude')
Coordinates:
  * latitude    (latitude) float64 264B 28.0 27.75 27.5 ... 20.5 20.25 20.0
  * longitude   (longitude) float64 200B 87.0 87.25 87.5 ... 92.5 92.75 93.0
  * valid_time  (valid_time) datetime64[ns] 91kB 1989-01-01 ... 2019-12-31
(11322, 33, 25)
float32
<xarray.DataArray 'tasmax' ()> Size: 4B
array(0)


## 3. Filtering dataset by time and location

We use `.sel` method to filter our data based on a specific or a range of time and/or location.

```python
# filtering by time
temp_df.sel(valid_time = "2000")
temp_df.sel(valid_time = "2000-07")

# filtering by time range
temp_df.sel(valid_time = slice("2000-01-01", "2000-12-31"))
```

In [28]:
cell = temp_df.sel(latitude=23.8, longitude=90.4, method="nearest")
# If we don't have the exact coordinates, we can use the method = "nearest" to select the nearest grid points.

# Check which coordinates were selected
print(cell.latitude.values, cell.longitude.values)

23.75 90.5


In [35]:
# First time step, first 10 latitude rows, first 15 longitude columns
subset = temp_df.isel(valid_time=0, latitude=slice(0, 10), longitude=slice(0, 15))

### 3.2 Some things about filter based on geographic region

>The direction of latitude coordinates matters. Some datasets store latitude from north to south (descending), while others store it south to north (ascending). Check the coordinate values first.

In [31]:
temp_df.latitude.values

array([28.  , 27.75, 27.5 , 27.25, 27.  , 26.75, 26.5 , 26.25, 26.  ,
       25.75, 25.5 , 25.25, 25.  , 24.75, 24.5 , 24.25, 24.  , 23.75,
       23.5 , 23.25, 23.  , 22.75, 22.5 , 22.25, 22.  , 21.75, 21.5 ,
       21.25, 21.  , 20.75, 20.5 , 20.25, 20.  ])

This dataset has a descending latitude.

In [46]:
# Example for descending latitude (common in some reanalysis files)
bd = temp_df.sel(
    latitude=slice(26.7, 20.5),
    longitude=slice(88.0, 92.8)
)
bd.latitude.values

array([26.5 , 26.25, 26.  , 25.75, 25.5 , 25.25, 25.  , 24.75, 24.5 ,
       24.25, 24.  , 23.75, 23.5 , 23.25, 23.  , 22.75, 22.5 , 22.25,
       22.  , 21.75, 21.5 , 21.25, 21.  , 20.75, 20.5 ])

In [45]:
# Example for ascending latitude
bd = ds.sel(
    latitude=slice(20.5, 26.7),
    longitude=slice(88.0, 92.8)
)

bd.latitude.values

array([], dtype=float64)